# Chapter 6: Evaluation

**Estimated time: 20 minutes**

You built a RAG pipeline. You ask it a question, it returns an answer, and it looks right. So you ship it. Two weeks later, support tickets roll in. The chatbot told a user they can cancel anytime, but the policy actually says thirty days notice with a processing fee. Your retriever pulled the wrong document, the LLM invented a detail, and nobody caught it.

"It seems to work" is not a shipping metric. In this chapter you stop eyeballing answers and start measuring them. Three numbers from RAGAS tell you whether your RAG is production ready. Faithfulness. Answer relevance. Context recall. Plus one more that catches retriever quality: context precision.

## What you will see in twenty minutes

You are going to take a five question test set over the SkillAgents AI corpus, run it through a retrieval pipeline, and score every answer with four RAGAS metrics. Then you are going to deliberately break one answer so you can watch the faithfulness score fall off a cliff.

Here is the arc:

- **Baseline run.** Five production style questions about refund policy, cohort schedule, payment errors, plan upgrades, and tax invoices. Each one has a ground truth answer. The retriever runs, the LLM answers, RAGAS scores every row. You read the scores as a pandas table with a red to green gradient.
- **Broken answer demo.** You replace one good answer with a hallucinated one ("full refund within thirty days, no fee, no questions asked"). You rerun RAGAS on that single row. Faithfulness collapses from something above 0.9 to something below 0.3. The other three metrics barely move. That is the whole point of having four metrics.
- **Interactive tweaks.** You swap the judge model from gpt-4o-mini to gpt-4o and rerun. You add a sixth question. You ship the whole thing as a script.

The lesson that matters is not the absolute number on any single row. It is the shape of the table. You want to walk away knowing which row to ship and which row to fix, and you want to know that without reading every answer yourself.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment. RAGAS depends on the huggingface datasets library at runtime, so both packages are in the install list.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 90 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4 \
        ragas==0.2.9 \
        datasets
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so the
# next import reads the freshly cloned file, not a stale copy from an earlier
# session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI, LangSmith, and Cohere keys out of Colab secrets and turned on LangSmith tracing for this notebook. LangSmith traces are helpful, but RAGAS makes dozens of LLM calls per evaluation row and the tracer log lines can drown out the RAGAS progress bar. The Chapter 6 helper turns tracing off for the duration of the RAGAS call and turns it back on when RAGAS is done, so everything else in the notebook still traces normally.

## Look at the corpus

The corpus is the same six documents from earlier chapters. Pricing, product guide, billing FAQ, error codes, refund policy, and one outdated blog post. Chapter 6 uses the boring, useful parts of this corpus. Refund rules. Plan upgrade prices. Error code fixes. The kind of questions a real user would actually ask a billing chatbot.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()
for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at the test set

A RAGAS evaluation needs three things per row. The user question. The ground truth answer a human manually verified. The list of documents the retriever should have pulled. You store these in a file once and reuse them every time something in the pipeline changes.

The Chapter 6 test set lives at `data/skillagents/ch6_test_set.json`. Five questions. Each question maps to one or two source files and has a hand written ground truth answer sourced directly from the corpus.

In [ ]:
import json
from pathlib import Path

test_set_path = Path("data/skillagents/ch6_test_set.json")
test_set = json.loads(test_set_path.read_text())

# test_set is a dict with a "questions" list. Each entry has id, question,
# ground_truth, and expected_sources.
questions = test_set["questions"]
print(f"Loaded {len(questions)} questions from {test_set_path.name}\n")
for q in questions:
    print(f"  [{q['id']}]  {q['question']}")
    print(f"      expected sources: {', '.join(q['expected_sources'])}")
    print()

Five questions is small on purpose. RAGAS makes roughly ten LLM calls per row (three for faithfulness, two for answer relevance, a few for context precision and context recall). Five rows is about a minute of wall time on gpt-4o-mini and costs roughly two cents per full run. Once you trust the shape of the scorecard, you grow the test set to twenty or thirty questions for real CI.

## Show the test set as a table

`show_test_set` formats the five questions as a pandas table so you can read them all in one glance. Students often skip reading a JSON dump but they always read a styled table.

In [ ]:
from utils.shared import show_test_set

show_test_set(questions)

## Metric one: faithfulness

Faithfulness asks one question. Does the answer actually come from the retrieved context? Or did the LLM invent a detail?

Here is the canonical failure mode. A user asks "What is the refund policy for annual plans?". Your retriever pulls the right page. The page says pro-rated refund minus two months used, minus a 50 dollar processing fee. The LLM generates "Full refund within thirty days, no questions asked." That sounds great. Except the source never said that. The LLM invented the thirty day window and dropped the processing fee. Every claim in a faithful answer has to trace back to the provided context. Every claim in an unfaithful answer does not.

You measure faithfulness by breaking the answer into individual claims, then checking each claim against the retrieved context. If nine claims out of ten are supported, your faithfulness is 0.9. If every claim is invented, your faithfulness is 0.0.

## Metric two: answer relevance

Answer relevance asks a different question. Does the answer actually address what the user asked?

A user asks "How do I set up SSO for my team?". The LLM returns a textbook definition of SSO, what the acronym stands for, and how it improves security and user experience. Technically accurate. But the user asked how to set it up, not what it is. The answer is factual and useless at the same time. That is a low answer relevance score, regardless of how faithful the content is to the source.

RAGAS measures this by generating a question from the answer, then comparing that generated question to the original. If someone reading only the answer would ask the same question the user asked, relevance is high. If the generated question is something completely different, relevance is low. Faithfulness and answer relevance are independent axes. A perfectly faithful answer can still miss the question.

## Metric three: context recall

Context recall is the retriever side of the story. Did you even pull back the right documents?

A user asks about enterprise system requirements. The ground truth answer names four things: RAM, CPU cores, operating system, and database version. Your retriever pulls three chunks. Two chunks cover RAM, CPU, and operating system. The database version chunk is missing entirely. Two out of four ground truth items are covered. Context recall is 0.5.

Half the information the user needs is not in the context. Even a perfect LLM cannot answer correctly if the right documents are not retrieved. This is why context recall matters more than any generation metric when your users start reporting "the answer is just wrong". The fix in that case is never a better prompt. It is better retrieval.

## Metric four: context precision

Context precision is the fourth RAGAS metric and it pairs with context recall. Recall asks "did you find the right documents". Precision asks "did you rank them at the top".

A retriever that returns the right chunk at rank five is technically successful at recall, but in production you probably only feed the top three chunks to the LLM. If the right chunk is at rank five, it never reaches the answer. Context precision captures this. It rewards retrievers that put the relevant chunks at the top of the list and penalizes retrievers that bury the right answer under five irrelevant neighbors.

You care about precision and recall together. High recall with low precision means you found the answer but a reranker might help. Low recall means you did not find the answer at all and a reranker cannot save you. Chapter 5 showed you how to fix low precision with reranking. This chapter shows you how to measure it in the first place.

## RAGAS in one cell

RAGAS is an open source evaluation framework that computes all four metrics for you. You feed it a list of samples, where each sample has a user question, the retrieved contexts, the generated answer, and the ground truth answer. You pick which metrics you care about, you pass in an LLM to act as the judge, and RAGAS returns a scored table.

Think of RAGAS as unit tests for your RAG pipeline. You would not ship a feature without tests. Do not ship RAG without evaluation.

The Chapter 6 helper `run_ragas_eval` hides the RAGAS plumbing. You pass it a pipeline function that takes a question and returns `(contexts, answer)`. You pass it a test set. It runs every question through the pipeline, hands the results to RAGAS, and returns a styled pandas table with a red to green gradient across the four metrics.

## Build the baseline pipeline

The pipeline is intentionally boring. Same FAISS index, same chunking strategy, same generation helper as earlier chapters. This is what a PM-level "v1 RAG" looks like before anyone has measured anything.

In [ ]:
from utils.shared import build_index, search, generate_answer

# Chapter 1 established that chunk_size=500 with chunk_overlap=50 is the
# right starting point for this corpus. We are holding chunking constant
# for this chapter so the only thing being measured is answer quality.
index = build_index(chunk_size=500, chunk_overlap=50)

def baseline_pipeline(question: str):
    """Run the baseline RAG pipeline and return (contexts, answer)."""
    results = search(index, question, k=3)
    contexts = [doc.page_content for doc, _ in results]
    answer = generate_answer(results, question)
    return contexts, answer

# Smoke test on a single question from the test set before running the full eval.
sample_q = questions[0]["question"]
sample_contexts, sample_answer = baseline_pipeline(sample_q)
print(f"Q: {sample_q}")
print(f"Answer:\n{sample_answer}")

Read the answer above. On gpt-4o-mini with this corpus, the baseline pipeline produces an answer that looks about right. It mentions the pro-rated calculation and the processing fee. That is the "it seems to work" moment. Time to measure.

## Run RAGAS on the full test set

`run_ragas_eval` takes the test set and the baseline pipeline, runs every question, and returns two things: a styled DataFrame for reading, and the raw DataFrame for further computation. Expect this cell to take about a minute. RAGAS is making roughly fifty LLM calls in the background, one batch per metric per row.

In [ ]:
from utils.shared import run_ragas_eval

styled_baseline, df_baseline = run_ragas_eval(questions, baseline_pipeline)
styled_baseline

Read the scorecard. The score columns go red to green. Green is good, red is bad. The four columns are the four metrics from the scenes above. Faithfulness and answer relevance are generation metrics, context precision and context recall are retrieval metrics.

What should you be looking at? Not the absolute numbers on any single row. The shape of the table. A row with green across all four is a row where the pipeline is doing its job end to end. A row with green on context precision and context recall but red on faithfulness is a generation problem (the LLM is making things up even though it has the right documents). A row with red on context recall is a retrieval problem (the LLM cannot possibly answer correctly because the right information was not pulled in). The diagnosis changes the fix.

## The overall scorecard

In production you track the average of each metric over time. Below are the four averages for your baseline run.

In [ ]:
from utils.shared import show_score_averages

show_score_averages(df_baseline, label="baseline")

These are the four numbers you would paste into a slack message after a model change. They compress five row scores into four averages you can eyeball in two seconds.

## What good looks like

RAGAS scores are between 0 and 1. Here are starter benchmarks for a PM:

- **Faithfulness.** Aim for 0.85 or higher. Below 0.7 means the LLM is regularly making up details. That is a support ticket factory.
- **Answer relevance.** Aim for 0.80 or higher. Below 0.6 means users are getting technically correct but useless answers.
- **Context recall.** Aim for 0.75 or higher. Below 0.5 means the retriever is missing half the relevant information. No LLM fixes bad retrieval.
- **Context precision.** Aim for 0.70 or higher. Below 0.5 means your top chunks are wrong even when the right chunk is somewhere in the pool. A reranker usually fixes this.

Green means ship it. Amber means investigate. Red means stop and fix before going live. These are starting points. Medical and legal applications need higher faithfulness. Search heavy products need higher context recall. Your exact thresholds depend on your domain and the cost of a wrong answer.

## Break one answer on purpose

The whole point of measuring is to catch problems before users do. Let us prove that RAGAS would catch a hallucination. You are going to take the plan upgrade question and replace the good baseline answer with a wrong one. The billing FAQ is unambiguous about the upgrade price. The student plan is 499 dollars, the Pro Annual plan is 1499 dollars, and the upgrade difference is 1000 dollars. The hallucinated answer invents a different number and a policy the doc does not mention. It is exactly the kind of confident wrong answer that slips past a human reviewer.

In [ ]:
# Test row index 3 is the plan upgrade question. We keep the retrieved
# contexts identical to the baseline run and only swap the answer. This
# isolates the faithfulness signal from the retrieval signal.
broken_question = questions[3]

# Run the retriever so we still have the real contexts for the ground truth.
baseline_contexts, _ = baseline_pipeline(broken_question["question"])

hallucinated_answer = (
    "To switch from the Student plan to Pro Annual on SkillAgents AI, you pay a "
    "250 dollar upgrade fee plus a 99 dollar admin charge. Your old cohort access "
    "is wiped and you must re-enroll from scratch. The upgrade takes up to seven "
    "business days to process."
)

print(f"Question: {broken_question['question']}")
print()
print("Retrieved contexts (unchanged):")
for i, c in enumerate(baseline_contexts, start=1):
    print(f"  [{i}] {c[:140]}...")
print()
print("Hallucinated answer:")
print(hallucinated_answer)

Read the contexts and the answer side by side. The billing FAQ says you pay the 1000 dollar difference between 499 and 1499, your cohort access does not reset, and the upgrade is instant from the billing page. The answer says a 250 dollar fee, a 99 dollar admin charge, a cohort reset, a re-enrollment, and a seven day processing delay. Every specific claim in the answer contradicts the retrieved context. A human skimming the answer might believe it because it sounds like generic SaaS billing copy. RAGAS reads the answer against the contexts and catches the lie.

## Score the broken answer

In [ ]:
from utils.shared import score_single_sample

# Run RAGAS on a one-row dataset: original question, real retrieved contexts,
# hallucinated answer, real ground truth. This scores the broken row in
# isolation so you can see how the numbers move.
broken_styled, broken_df = score_single_sample(
    question=broken_question["question"],
    contexts=baseline_contexts,
    answer=hallucinated_answer,
    ground_truth=broken_question["ground_truth"],
)
broken_styled

Look at the four scores. Context precision and context recall should still be high or perfect, because the retriever did its job. Faithfulness should collapse, because none of the claims in the answer trace back to the contexts. Answer relevance can stay middling to high, because the hallucinated answer is still on-topic for the question. That last observation is the whole argument for measuring faithfulness separately. A wrong answer can still sound relevant. Only a faithfulness check catches the lie.

This is the result you would never want to see in production and the exact result you want your eval suite to catch. A pull request that introduces a prompt change dropping faithfulness from 0.9 to 0.2 should fail CI. The test set in this notebook is small, but the shape is identical to what a production CI eval would run.

## Building a real test suite

You have the loop now. Here is how to scale it.

- **Start with 20 to 30 golden question and answer pairs.** These are questions your users actually ask, paired with answers you have manually verified against the source docs. More rows means more stable averages. Five is a demo. Twenty is a signal. Fifty is a production gate.
- **Store them in one file.** JSON or CSV. The format does not matter. What matters is that every PM, engineer, and prompt writer reads from the same file. If the test set lives in one person's head, the eval does not exist.
- **Run the test set on every change.** Every time you change your chunking, swap an embedding model, rewrite a prompt, or upgrade an LLM, re-run the suite. Store the scores with a git SHA so you can tell which commit moved the numbers.
- **Revert when scores drop.** This is the discipline that turns RAG from a demo into a product. A change that improves answer relevance but drops faithfulness is almost never worth shipping.

## Interactive: swap the judge model

The LLM that scores a RAGAS run is called the judge. By default you pass gpt-4o-mini because it is cheap. On hard domains you sometimes want a stronger judge, even though the cost per evaluation goes up. `run_ragas_eval` takes a `judge_model` argument. Try swapping it.

In [ ]:
# Change judge_model to "gpt-4o" and rerun to see a stronger judge in action.
# The default is "gpt-4o-mini". gpt-4o costs roughly 10x more per eval but
# gives more consistent faithfulness scores on long answers.
judge_model = "gpt-4o-mini"  # Try changing to "gpt-4o"

styled_swap, df_swap = run_ragas_eval(
    questions[:2],  # Only two rows so the swap is fast.
    baseline_pipeline,
    judge_model=judge_model,
)
styled_swap

Compare the two rows above to the same two rows in the baseline scorecard. If you stayed on gpt-4o-mini the numbers will be almost identical. If you switched to gpt-4o the numbers usually tighten up a little, especially on faithfulness. The judge is not a free lunch. A stronger judge does not improve your pipeline, it only measures it more precisely. If your production scores are noisy run to run, a stronger judge is the fix. If your production scores are stable but low, a stronger judge will not save you. Fix the pipeline.

## Open your LangSmith trace

RAGAS turns LangSmith tracing off during the eval call because the ragas internals make dozens of calls per row and the log spam drowns out everything else. Every call outside of `run_ragas_eval` still traces normally. Open [smith.langchain.com](https://smith.langchain.com), pick the `rag-for-pms` project, and click into the most recent trace to see the generation calls from the baseline pipeline. You will not see the RAGAS internal calls there. That is by design.

## What you can do at work on Monday

Evaluation is the single most underused discipline in applied RAG. Most teams ship on vibes and learn about problems from support tickets. Here are the three questions to ask on your next RAG review.

1. What is your test set? If the answer is "we test it manually every few weeks", you do not have a test set. A test set is a file with golden question and answer pairs that gets rerun on every change.
2. What is the faithfulness score of your pipeline today, and what was it on the last shipped version? If nobody can answer, nobody is measuring. If the score moved, somebody should be able to explain why.
3. When a user reports a wrong answer, do you add that question to the test set before you fix the bug? Every production bug should become a new row in the test set so the same bug cannot ship twice.

A team that cannot answer these questions is running on vibes. Evaluation is what turns a demo into a product.

## Try it yourself

Pick any of these. Change the value in the relevant cell, rerun, watch what happens.

1. **Add a sixth question.** Append a new entry to `questions` in memory. Pick a real question about the corpus. Write the ground truth yourself. Rerun `run_ragas_eval`. Your sixth row appears at the bottom of the table with four scores.
2. **Change `k=3` to `k=1` in `baseline_pipeline`.** With only one retrieved chunk, context recall will probably drop on questions that need two sources (like the E-4012 payment question). The other metrics may or may not move. This is how you confirm that context recall is measuring what you think it is measuring.
3. **Rewrite one ground truth on purpose.** Replace the refund policy ground truth with a version that says "no refunds ever" (a wrong ground truth). Watch context recall drop, because the retriever still brings back the real policy but RAGAS now thinks the real policy does not match the ground truth. This shows why your test set quality matters more than your pipeline quality. Bad ground truth means every score is lying to you.

## Homework

Two exercises you can do on your own. Each takes about fifteen minutes.

1. **Write your own three question test set.** Pick a real product from your own company. Write three questions a user would actually ask. For each one, find the correct answer in your own docs and write a ground truth. Save it as a JSON file in the same shape as `ch6_test_set.json`. Point `run_ragas_eval` at it. Read the scores. The first time you do this for your own product is the moment RAG stops being abstract.

2. **Ship the eval as a CI script.** Take the cells in this notebook from `baseline_pipeline` through `run_ragas_eval` and copy them into a plain Python file called `eval.py`. Add a line at the bottom that prints the averages and exits with a non-zero exit code if any average is below your threshold. Commit it. Next time you change a prompt or a chunking parameter, run `python eval.py` before merging. This is the discipline difference between "we tested it once" and "we test it every time".

## What is next

You now know how to measure a RAG pipeline. You can run a test set, read the scores, spot regressions, and reject bad prompt changes before they ship. The next chapter closes the loop. Instead of having a human read the scores and decide what to do, the pipeline itself checks its own answer, notices when the answer is not grounded in the retrieved context, rewrites the query, and retries. Evaluation becomes part of generation. See you in Chapter 7.